# 🚀 Launch Thermal DIP Backend API Server in Google Colab
### Module 03: FastAPI REST Server with Free HTTPS Tunnel
This notebook launches the FastAPI backend on the Google Colab T4 GPU, exposing:
- `POST /api/restorations` (Submit restoration job)
- `GET /api/restorations/{id}/status` (Poll job progress & preview)
- `GET /api/restorations/{id}/result` (Retrieve final restored image + metrics)
- `POST /api/restorations/{id}/cancel` (Cancel running job)
- `GET /api/health` (Health & GPU inspection)

It generates a **public HTTPS URL** via `cloudflared` or `pyngrok` so you can connect the frontend web studio from anywhere.

In [ ]:
# 1. GPU Check & Dependencies
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

from pathlib import Path
if not Path("ml/dip_core").exists():
    !git clone https://github.com/likhith-naik01/DIP_MP.git
    %cd DIP_MP

# Install ML package and backend dependencies
!pip install -q -e ml/
!pip install -q fastapi uvicorn python-multipart pydantic

In [ ]:
# 2. Download and Setup Cloudflared Tunnel (No account or token required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# 3. Launch Backend and Tunnel in Background
import subprocess
import time
import re

# Start cloudflared tunnel on port 8000
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                               stdout=tunnel_log, stderr=tunnel_log)

# Wait 5 seconds for tunnel URL to generate
time.sleep(5)
public_url = None
with open("tunnel.log", "r") as f:
    for line in f:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

print("="*60)
if public_url:
    print(f"🎉 PUBLIC BACKEND URL: {public_url}")
    print(f"👉 Copy this URL into the frontend studio API Settings field!")
    print(f"API Health Endpoint: {public_url}/api/health")
    print(f"API Docs: {public_url}/docs")
else:
    print("Could not parse cloudflared URL yet. Check tunnel.log.")
print("="*60)

In [ ]:
# 4. Start FastAPI Uvicorn Server (Runs in Foreground)
# Set PYTHONPATH so backend and ml packages can be imported cleanly
import os
os.environ["PYTHONPATH"] = f"{os.getcwd()}:{os.getcwd()}/ml:{os.getcwd()}/backend"

!uvicorn backend.app.main:app --host 0.0.0.0 --port 8000